# LangGraph Customer Support Agent with Mem0 Memory

A customer-support chatbot built as a LangGraph `StateGraph` whose node reads from and writes to Mem0. Based on the [Mem0 LangGraph guide](https://docs.mem0.ai/integrations/langgraph).

In [ ]:
%pip install -qU langgraph langchain-openai mem0ai python-dotenv

## API keys

Reads `OPENAI_API_KEY` and `MEM0_API_KEY` from `.env` / the environment, and prompts only if missing.

In [ ]:
import os, getpass
from dotenv import load_dotenv

load_dotenv()

for key in ("OPENAI_API_KEY", "MEM0_API_KEY"):
    if not os.environ.get(key):
        os.environ[key] = getpass.getpass(f"{key}: ")

In [ ]:
from typing import Annotated, TypedDict, List

from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from mem0 import MemoryClient

## Clients and state

The state carries the message list plus the Mem0 user id, so the node knows whose memories to use.

In [ ]:
llm = ChatOpenAI(model="gpt-5-nano")
mem0 = MemoryClient()


class State(TypedDict):
    messages: Annotated[List[HumanMessage | AIMessage], add_messages]
    mem0_user_id: str

## The chatbot node

Search Mem0 for relevant memories, put them in the system message, call the LLM, then save the exchange.

In [ ]:
def chatbot(state: State):
    messages = state["messages"]
    user_id = state["mem0_user_id"]

    try:
        memories = mem0.search(messages[-1].content, filters={"user_id": user_id})
        context = "Relevant information from previous conversations:\n"
        for memory in memories["results"]:
            context += f"- {memory['memory']}\n"

        system_message = SystemMessage(content=(
            "You are a helpful customer support assistant. Use the provided context "
            "to personalize your responses and remember user preferences and past "
            f"interactions.\n{context}"
        ))
        response = llm.invoke([system_message] + messages)

        try:
            result = mem0.add(
                [
                    {"role": "user", "content": messages[-1].content},
                    {"role": "assistant", "content": response.content},
                ],
                user_id=user_id,
            )
            print(f"[Memory saved: {len(result.get('results', []))} memories added]")
        except Exception as e:
            print(f"Error saving memory: {e}")

        return {"messages": [response]}
    except Exception as e:
        print(f"Error in chatbot: {e}")
        return {"messages": [llm.invoke(messages)]}

## Build the graph

The Mem0 guide routes `chatbot` back to itself, which would loop until the recursion limit; here the node ends the run after one reply.

In [ ]:
graph = StateGraph(State)
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)
compiled_graph = graph.compile()

compiled_graph

## Try it

Share some details, then ask a question that relies on them.

In [ ]:
def run_conversation(user_input: str, mem0_user_id: str):
    state = {"messages": [HumanMessage(content=user_input)], "mem0_user_id": mem0_user_id}
    result = compiled_graph.invoke(state)
    print("Customer Support:", result["messages"][-1].content)


mem0_user_id = "alice"

for msg in [
    "Hi, I ordered a laptop last week (order #4821) and it arrived with a cracked screen.",
    "What was my order number again, and what was wrong with it?",
]:
    print("You:", msg)
    run_conversation(msg, mem0_user_id)
    print()

## Interactive chat (optional)

Run this cell to chat; type `quit` to stop.

In [ ]:
while True:
    user_input = input("You: ")
    if user_input.lower() in ("quit", "exit", "bye"):
        print("Customer Support: Thank you for contacting us. Have a great day!")
        break
    run_conversation(user_input, mem0_user_id)